Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, Trainer, TrainingArguments
from torch.utils.data import DataLoader
from transformers import AdamW, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import optuna
from transformers import EvalPrediction, get_linear_schedule_with_warmup
import torch
import numpy as np
import pandas as pds
import warnings
warnings.filterwarnings('ignore')

This notebook was run on google colab because of the GPU support.

This workflow aim to solve the paraphrase classification task via finetuning a pretrained BERT model. 
Following steps are involved in this workflow:
1. Load the dataset
2. Preprocess the dataset so they can be fed into BERT
3. Utilize the Hugging Face Transformers library to load a pretrained BERT model
4. Finetune the model on the dataset and conduct hyperparameter search to find the best model
5. Evaluate the best model on the test set

In [ ]:
# read from train/test data files using the modified code as baseline_logistic model
def readInData(filename):

    data = []
    trends = set([])

    (trendid, trendname, origsent, candsent, judge, origsenttag, candsenttag) = (None, None, None, None, None, None, None)

    for line in open(filename):
        line = line.strip()
        #read in training or dev data with labels
        if len(line.split('\t')) == 7:
            (trendid, trendname, origsent, candsent, judge, origsenttag, candsenttag) = line.split('\t')
        #read in test data without labels
        elif len(line.split('\t')) == 6:
            (trendid, trendname, origsent, candsent, origsenttag, candsenttag) = line.split('\t')
        else:
            continue

        #if origsent == candsent:
        #    continue

        trends.add(trendid)

        if judge == None:
            data.append(( judge, origsent, candsent, trendid))
            continue

        # ignoring the training/test data that has middle label
        if judge[0] == '(':  # labelled by Amazon Mechanical Turk in format like "(2,3)"
            nYes = eval(judge)[0]
            if nYes >= 3:
                amt_label = True
                data.append((amt_label, origsent, candsent, trendid))
            elif nYes <= 1:
                amt_label = False
                data.append((amt_label, origsent, candsent, trendid))
        elif judge[0].isdigit():   # labelled by expert in format like "2"
            nYes = int(judge[0])
            if nYes >= 4:
                expert_label = True
                data.append(( expert_label, origsent, candsent, trendid))
            elif nYes <= 2:
                expert_label = False
                data.append(( expert_label, origsent, candsent, trendid))
            else:
                expert_label = None
                data.append(( expert_label, origsent, candsent, trendid))
        else:
            continue

    return data, trends


# ParaphraseDataset class
class ParaphraseDataset():
    def __init__(self, data, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.sentences_pairs = []
        self.labels = []

        # Load data from file

        for item in data:
            judge, orig_sent, cand_sent, trend_id = item
            self.sentences_pairs.append((orig_sent, cand_sent))
            self.labels.append(int(judge))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        orig_sent, cand_sent = self.sentences_pairs[item]
        label = self.labels[item]

        # Encode the pair of sentences to be fed to BERT
        encoding = self.tokenizer.encode_plus(
            orig_sent,
            cand_sent,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Compute evaluation metrics for the model
def compute_metrics(p: EvalPrediction):
    preds = np.argmax(p.predictions, axis=1)
    return {"eval_accuracy": accuracy_score(p.label_ids, preds)}

# using optima to search for the best hyperparameters
def objective(trial):
    learning_rate = trial.suggest_loguniform('lr', 1e-5, 5e-5)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    num_train_epochs = trial.suggest_int('epochs', 2, 5, 7)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    # Configure the TrainingArguments
    training_args = TrainingArguments(
        output_dir=f'{project_path}/results/trial_{trial.number}',  
        num_train_epochs=num_train_epochs,  
        per_device_train_batch_size=batch_size,  
        per_device_eval_batch_size=batch_size,  
        warmup_steps=500,  
        weight_decay=0.01,  # adding some regularization
        logging_dir=f'{project_path}/logs/trial_{trial.number}',  
        learning_rate=learning_rate,
        evaluation_strategy="epoch",  # evaluate at the end of each epoch
        save_strategy="epoch",  
        load_best_model_at_end=True,  # load the best model at the end of training
        metric_for_best_model="eval_accuracy"  # use the evaluation accuracy as the metric to compare between models
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset= train_paraphrased,
        eval_dataset= val_paraphrased,
        compute_metrics = compute_metrics
    )

    trainer.train()
    evaluation_result = trainer.evaluate()

    return evaluation_result["eval_accuracy"]



In [ ]:
# specify the path to the data files
data_root = Path("/content/drive/MyDrive/")

project_name = "grasp-data-hometask-semantic-similarity-master"
project_path = data_root / project_name
train_data_path = project_path / "data/train.data"
train_data, train_trend = readInData(train_data_path)

# remove the data that has None in the label for the training data
filtered_train_data = [item for item in train_data if None not in item]

# split the training data into training and validation data
train_data, val_data = train_test_split(filtered_train_data, test_size=0.2, random_state=42)

test_data_path = project_path / "data/test.data"

# read in the test data
test_data, test_trend = readInData(test_data_path)


In [ ]:
# Load the BERT tokenizer and create training and validation datasets in the ParaphraseDataset class
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_paraphrased = ParaphraseDataset(train_data, tokenizer, max_len=40)
val_paraphrased = ParaphraseDataset(val_data, tokenizer, max_len=40)


In [ ]:
# Load the BERT model and optimizer
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)  # 2 labels: 0 and 1 for paraphrase and non-paraphrase

# Conduct hyperparameter optimization using Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Get the best trial
best_trial = study.best_trial
print(f'Best trial final validation accuracy: {best_trial.value:.4f}')
print(f'Best trial hyperparameters: {best_trial.params}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2024-08-05 19:51:58,874] A new study created in memory with name: no-name-81c85b49-07eb-410a-96c8-aec99fbffbc8
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.403461,0.824577
2,0.476800,0.304668,0.885367


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[I 2024-08-05 19:53:08,822] Trial 0 finished with value: 0.8853669127225359 and parameters: {'lr': 2.2563850057358655e-05, 'batch_size': 32, 'epochs': 2}. Best is trial 0 with value: 0.8853669127225359.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.506400,0.350244,0.852366
2,0.306900,0.290478,0.883196


...

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[I 2024-08-05 20:28:53,268] Trial 28 finished with value: 0.8862353452019105 and parameters: {'lr': 4.1236297183055145e-05, 'batch_size': 16, 'epochs': 2}. Best is trial 26 with value: 0.8918801563178463.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.484200,0.343694,0.854103
2,0.307600,0.283867,0.885801


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[I 2024-08-05 20:30:16,432] Trial 29 finished with value: 0.8858011289622232 and parameters: {'lr': 4.509718186348623e-05, 'batch_size': 16, 'epochs': 2}. Best is trial 26 with value: 0.8918801563178463.


Best trial final validation accuracy: 0.8919
Best trial hyperparameters: {'lr': 4.2701475542552015e-05, 'batch_size': 16, 'epochs': 2}


In [ ]:
# Load the best model and evaluate it on the test data

def OutputPredictions(modelfile, test_data, outfile):
    # Load the best fine-tuned model
    model = BertForSequenceClassification.from_pretrained(modelfile)
    model.eval()  # set the model to evaluation mode
    outf = open(outfile, 'w')

    with torch.no_grad():
        for i, (_, text1, text2, _) in enumerate(test_data):
                encoded_dict = tokenizer.encode_plus(
                    text1, text2,                      
                    add_special_tokens=True,           
                    max_length=64,                     # Pad & truncate sentences.
                    pad_to_max_length=True,
                    return_attention_mask=True,        
                    return_tensors='pt',               
                )

                input_ids = encoded_dict['input_ids']
                attention_mask = encoded_dict['attention_mask']

                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs[0]
                probabilities = torch.softmax(logits, dim=1)
                prob = probabilities[:, 1].item()  # obtain probabiilty of the positive class

                if prob >= 0.5:
                    outf.write("true\t" + "{0:.4f}".format(prob) + "\n")
                else:
                    outf.write("false\t" + "{0:.4f}".format(prob) + "\n")


# test the best model on the test data
model_path = f'{project_path}/results/trial_{best_trial.number}/checkpoint-576/'
OutputPredictions(model_path, test_data, f'{project_path}/systemoutputs/PIT2015_BERT_03_BCL.output')